# LeetCode 677 — Map Sum Pairs bằng Python

Link đề: https://leetcode.com/problems/map-sum-pairs/ · Độ khó: **Medium** · Chủ đề: Design (thiết kế lớp), Trie (cây tiền tố), Hash Table, String

Skeleton LeetCode:

```python
class MapSum:

    def __init__(self):
        ...

    def insert(self, key: str, val: int) -> None:
        ...

    def sum(self, prefix: str) -> int:
        ...


# Your MapSum object will be instantiated and called as such:
# obj = MapSum()
# obj.insert(key,val)
# param_2 = obj.sum(prefix)
```

Ghi chú: đây là bài **thiết kế cấu trúc dữ liệu**. LeetCode chạy một chuỗi lệnh (`["MapSum","insert","sum",...]`) lên đối tượng của bạn; notebook này tự dựng một "bộ chạy chuỗi lệnh" y như vậy để kiểm tra mọi cách giải.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một "bản đồ" `key -> val` với hai thao tác `insert` (ghi, **ghi đè** nếu key đã có) và `sum(prefix)` (tổng giá trị của mọi key bắt đầu bằng `prefix`).
2. Nắm các công cụ Python cần cho bài thiết kế: lớp và `__init__` (trạng thái riêng của từng đối tượng), `dict`, `defaultdict`, `str.startswith`, cắt chuỗi `key[:i]`, `ord/chr`, `bisect_left`.
3. Hiểu **Trie** (cây tiền tố) và mẹo cốt lõi: lưu **tổng của cả cây con** ở mỗi nút, và khi ghi đè chỉ cộng **chênh lệch** `delta = val_mới - val_cũ`.
4. Đi qua 5 cách: quét mọi key, bảng tổng theo từng tiền tố, Trie lưu tổng (delta), Trie lưu giá trị ở nút cuối + duyệt cây con, danh sách đã sắp xếp + `bisect`; so sánh được chi phí `insert` vs `sum` của từng cách.
5. Tự dựng bộ chạy chuỗi lệnh để test như LeetCode, và tránh các lỗi kinh điển: quên trừ giá trị cũ khi ghi đè, `in` thay `startswith`, chốt chặn `bisect` sai, trạng thái dùng chung giữa các đối tượng, truy cập nút `None` trong Trie.

## 2. Đề bài & ví dụ

Thiết kế một map cho phép:

- Ánh xạ một chuỗi `key` tới một giá trị `val`.
- Trả về **tổng** các giá trị của mọi cặp mà `key` có **tiền tố** (prefix) bằng một chuỗi cho trước.

Cài đặt lớp `MapSum`:

- `MapSum()` khởi tạo đối tượng.
- `void insert(String key, int val)` thêm cặp `key-val`. Nếu `key` đã tồn tại, giá trị cũ bị **ghi đè** bằng giá trị mới.
- `int sum(string prefix)` trả về tổng giá trị của mọi cặp có `key` bắt đầu bằng `prefix`.

**Ví dụ 1**

```text
Input
["MapSum", "insert", "sum", "insert", "sum"]
[[], ["apple", 3], ["ap"], ["app", 2], ["ap"]]
Output
[null, null, 3, null, 5]
```

- `insert("apple", 3)`; `sum("ap")` -> 3 (chỉ có `apple`).
- `insert("app", 2)`; `sum("ap")` -> 5 (`apple` + `app` = 3 + 2).

**Ràng buộc:**

- `1 <= key.length, prefix.length <= 50`
- `key` và `prefix` chỉ gồm chữ thường tiếng Anh.
- `1 <= val <= 1000`
- Tối đa `50` lần gọi `insert` và `sum`.

Lưu ý: một key cũng là tiền tố của chính nó (`sum("app")` tính cả key `app`); `sum` một tiền tố không tồn tại trả `0`.

In [1]:
from typing import List, Optional
from collections import defaultdict
from bisect import bisect_left, insort
import random, time

def run_ops(cls, ops: List[str], args: list) -> list:
    """Mô phỏng cách LeetCode gọi: phần tử đầu là tên lớp (constructor),
    các phần tử sau là tên method. Trả về list kết quả, None ứng với null."""
    obj = None
    out = []
    for i, (op, arg) in enumerate(zip(ops, args)):
        if i == 0:
            assert op == cls.__name__, (op, cls.__name__)
            obj = cls(*arg)
            out.append(None)
        else:
            out.append(getattr(obj, op)(*arg))
    return out

example_ops = ["MapSum", "insert", "sum", "insert", "sum"]
example_args = [[], ["apple", 3], ["ap"], ["app", 2], ["ap"]]
example_expected = [None, None, 3, None, 5]
print("ops     =", example_ops)
print("kỳ vọng =", example_expected)

ops     = ['MapSum', 'insert', 'sum', 'insert', 'sum']
kỳ vọng = [None, None, 3, None, 5]


## 3. Kiến thức nền

### 3.1. Lớp và trạng thái riêng của từng đối tượng

Trong bài thiết kế, LeetCode tạo `MapSum()` rồi gọi liên tiếp các method lên **cùng một đối tượng**. Dữ liệu phải được cất trong `self` (thuộc tính của đối tượng, tạo ở `__init__`) để các lần gọi sau nhìn thấy kết quả lần trước. Nhiều test case trong một lần chấm tạo **nhiều đối tượng riêng**, nên trạng thái **không được** đặt ở cấp lớp (class attribute) vì sẽ bị dùng chung.

In [2]:
class Counter1:
    def __init__(self):
        self.n = 0            # riêng cho từng đối tượng
    def inc(self):
        self.n += 1
        return self.n

a, b = Counter1(), Counter1()
print(a.inc(), a.inc(), b.inc())     # 1 2 1  (a và b độc lập)

# Chạy bằng bộ run_ops: phần tử đầu là constructor, null là None
print(run_ops(Counter1, ["Counter1", "inc", "inc"], [[], [], []]))

1 2 1
[None, 1, 2]


### 3.2. `dict`, `.get`, `defaultdict` và ghi đè

`d[key] = val` **ghi đè** nếu `key` đã có. `d.get(key, 0)` đọc giá trị, nếu không có trả `0` (không tạo phần tử mới). `defaultdict(int)` tự tạo phần tử `0` khi truy cập key chưa có (tiện nhưng **truy cập cũng tạo phần tử**).

In [3]:
d = {}
d["apple"] = 3
d["apple"] = 10                       # ghi đè, không phải cộng thêm
print(d, d.get("pear", 0), "pear" in d)

dd = defaultdict(int)
dd["x"] += 5
print(dd["x"], dd["chưa có"], dict(dd))   # đọc key chưa có cũng làm nó xuất hiện

{'apple': 10} 0 False
5 0 {'x': 5, 'chưa có': 0}


### 3.3. Tiền tố: `startswith` và cắt chuỗi

`key.startswith(prefix)` kiểm tra `key` có bắt đầu bằng `prefix` không (khác với `prefix in key`, kiểm tra **bất kỳ vị trí**). `key[:i]` là tiền tố gồm `i` ký tự đầu. Một key độ dài `L` có đúng `L` tiền tố khác rỗng.

In [4]:
key = "apple"
print([key[:i] for i in range(1, len(key) + 1)])
print(key.startswith("ap"), key.startswith("pp"), "pp" in key)    # True False True  <- chú ý hai kết quả cuối
print("apple".startswith("apple"), "app".startswith("apple"))     # True False

['a', 'ap', 'app', 'appl', 'apple']
True False True
True False


### 3.4. Trie (cây tiền tố) — bản nhỏ bằng `dict`

Trie là cây mà mỗi **cạnh** là một ký tự, mỗi **đường từ gốc** ứng với một tiền tố. Các key chung tiền tố dùng chung đường đi, nên tìm "mọi key bắt đầu bằng `ap`" = đi theo `a`, `p` rồi nhìn xuống cây con.

Dưới đây là Trie thô: mỗi nút là một `dict` ký tự -> nút con; ta dùng khóa đặc biệt `"$"` đánh dấu "key kết thúc ở đây".

In [5]:
def trie_insert(root: dict, word: str):
    node = root
    for ch in word:
        node = node.setdefault(ch, {})
    node["$"] = True

root = {}
for w in ["app", "apple", "apt", "bat"]:
    trie_insert(root, w)

def show(node, indent=0):
    for ch, child in sorted(node.items()):
        if ch == "$":
            print("  " * indent + "(kết thúc)")
        else:
            print("  " * indent + ch)
            show(child, indent + 1)
show(root)

a
  p
    p
      (kết thúc)
      l
        e
          (kết thúc)
    t
      (kết thúc)
b
  a
    t
      (kết thúc)


### 3.5. Sắp xếp chuỗi và `bisect_left` (dùng ở Cách 5)

Chuỗi so sánh theo **thứ tự từ điển**. Mọi key bắt đầu bằng `prefix` nằm **liền nhau** trong danh sách đã sắp xếp, trong khoảng `[prefix, prefix + chốt chặn)`. Vì chữ cái lớn nhất là `'z'` (mã 122) mà `'{'` có mã 123 (ngay sau `'z'`), mọi key có tiền tố `prefix` đều **nhỏ hơn** `prefix + '{'`, và mọi key lớn hơn hoặc bằng nó thì không có tiền tố đó. `bisect_left(a, x)` cho vị trí đầu tiên mà `a[pos] >= x`.

In [6]:
keys = sorted(["apple", "app", "apt", "bat", "ap", "azz", "apzz"])
print(keys)
prefix = "ap"
lo = bisect_left(keys, prefix)
hi = bisect_left(keys, prefix + "{")
print("lo, hi =", lo, hi, "->", keys[lo:hi])
print(ord("z"), ord("{"), "apzz" < "ap" + "{", "apzz" < "ap" + "z")   # 'z' làm hỏng chốt chặn (apzz không nhỏ hơn 'apz')

['ap', 'app', 'apple', 'apt', 'apzz', 'azz', 'bat']
lo, hi = 0 5 -> ['ap', 'app', 'apple', 'apt', 'apzz']
122 123 True False


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ của đề:

1. `insert("apple", 3)`: map = `{apple: 3}`.
2. `sum("ap")`: key nào bắt đầu bằng `ap`? chỉ `apple` -> **3**.
3. `insert("app", 2)`: map = `{apple: 3, app: 2}`.
4. `sum("ap")`: `apple` và `app` -> 3 + 2 = **5**.

Bây giờ thử **ghi đè**: `insert("apple", 10)` -> map = `{apple: 10, app: 2}`, `sum("ap") = 12` (không phải 3 + 10 + 2). Nếu ta duy trì sẵn "tổng theo tiền tố", ghi đè phải **cộng thêm chênh lệch** `10 - 3 = +7` vào tổng mọi tiền tố của `apple` (`a`, `ap`, `app`, `appl`, `apple`): `sum("ap")` đi từ 5 lên 12.

In [7]:
m = {}
def hand_sum(prefix):
    return sum(v for k, v in m.items() if k.startswith(prefix))

m["apple"] = 3;  print("sum('ap') =", hand_sum("ap"))
m["app"] = 2;    print("sum('ap') =", hand_sum("ap"))
old = m["apple"]
m["apple"] = 10
print("ghi đè apple 3 -> 10: delta =", 10 - old, "| sum('ap') =", hand_sum("ap"), "(= 5 + delta)")
print("sum('') =", hand_sum(""), "(tiền tố rỗng: mọi key; ngoài ràng buộc của đề)  | sum('z') =", hand_sum("z"))

sum('ap') = 3
sum('ap') = 5
ghi đè apple 3 -> 10: delta = 7 | sum('ap') = 12 (= 5 + delta)
sum('') = 12 (tiền tố rỗng: mọi key; ngoài ràng buộc của đề)  | sum('z') = 0


## 5. Cách 1 — Brute Force: lưu dict, mỗi lần `sum` quét mọi key

**Ý tưởng.** Cất `key -> val` trong một `dict`. `insert` chỉ là `d[key] = val` (tự ghi đè). `sum(prefix)` duyệt **mọi** cặp, cộng giá trị nếu `key.startswith(prefix)`.

Đây là cách "đúng hiển nhiên" nên ta cũng dùng nó làm một trong các **đối chứng**.

In [8]:
class MapSum:
    def __init__(self):
        self.d = {}

    def insert(self, key: str, val: int) -> None:
        self.d[key] = val

    def sum(self, prefix: str) -> int:
        return sum(v for k, v in self.d.items() if k.startswith(prefix))

MapSumScan = MapSum
res = run_ops(MapSumScan, example_ops, example_args)
print(res)
assert res == example_expected

[None, None, 3, None, 5]


**Độ phức tạp và nút thắt.** Gọi `N` là số key phân biệt, `L` là độ dài prefix. `insert`: `O(L)` (băm chuỗi). `sum`: `O(N * L)` vì quét hết `N` key, mỗi key so sánh tối đa `L` ký tự. Bộ nhớ `O(tổng độ dài key)`. Với đề (50 lệnh) thì đủ nhanh, nhưng nếu có hàng trăm nghìn key và nhiều truy vấn `sum` thì quá chậm. Nút thắt: **mỗi truy vấn đều làm lại việc cộng**. Ta muốn "cộng sẵn".

## 6. Ý tưởng thuật toán / bước đột phá

**Cộng sẵn tổng theo tiền tố.** Với mỗi tiền tố `p` có thể gặp, duy trì `S[p]` = tổng giá trị của mọi key bắt đầu bằng `p`. Khi đó `sum(prefix)` chỉ là tra cứu `S[prefix]`.

**Cập nhật bằng delta.** Khi `insert(key, val)`:

- `delta = val - (giá trị cũ của key, hoặc 0 nếu key mới)`.
- Cộng `delta` vào `S[p]` cho **mọi tiền tố `p` của `key`** (đúng `len(key)` tiền tố).

Cách này xử lý ghi đè một cách tự nhiên mà không cần xóa gì.

Hai cách lưu `S`:

- bảng băm `tiền tố -> tổng` (Cách 2) — mỗi tiền tố là một chuỗi riêng, tốn bộ nhớ lặp ký tự;
- **Trie**: nút ứng với tiền tố `p` giữ `S[p]`, các tiền tố dùng chung ký tự (Cách 3).

Ngoài ra còn hai hướng khác không cộng sẵn: Trie chỉ lưu giá trị ở nút cuối rồi duyệt cây con (Cách 4), hoặc giữ key đã sắp xếp rồi dùng nhị phân cắt ra một đoạn liên tiếp (Cách 5).

| Cách | Ý tưởng |
|---|---|
| 2 | Bảng băm `tiền tố -> tổng`, cập nhật theo delta |
| 3 | Trie, mỗi nút giữ tổng cây con, cập nhật theo delta |
| 4 | Trie lưu giá trị ở nút cuối; `sum` duyệt cây con |
| 5 | Danh sách key đã sắp xếp; `bisect` tìm đoạn; cộng đoạn đó |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Bảng băm tổng theo tiền tố

**Giải thích từng đoạn.**

- `self.d`: giá trị hiện tại của từng key (để biết giá trị cũ khi ghi đè).
- `self.pre`: `tiền tố -> tổng`.
- `insert`: tính `delta`, ghi `d[key] = val`, rồi với `i` từ 1 tới `len(key)`: `pre[key[:i]] += delta`.
- `sum`: `pre.get(prefix, 0)`. Dùng `.get` (không dùng `pre[prefix]`) để truy vấn không tạo thêm phần tử.

In [9]:
class MapSum:
    def __init__(self):
        self.d = {}
        self.pre = defaultdict(int)

    def insert(self, key: str, val: int) -> None:
        delta = val - self.d.get(key, 0)
        self.d[key] = val
        for i in range(1, len(key) + 1):
            self.pre[key[:i]] += delta

    def sum(self, prefix: str) -> int:
        return self.pre.get(prefix, 0)

MapSumPrefixDict = MapSum
res = run_ops(MapSumPrefixDict, example_ops, example_args)
print(res)
assert res == example_expected

[None, None, 3, None, 5]


**Code nộp LeetCode (Cách 2):**

In [10]:
from collections import defaultdict

class MapSum:

    def __init__(self):
        self.d = {}
        self.pre = defaultdict(int)

    def insert(self, key: str, val: int) -> None:
        delta = val - self.d.get(key, 0)
        self.d[key] = val
        for i in range(1, len(key) + 1):
            self.pre[key[:i]] += delta

    def sum(self, prefix: str) -> int:
        return self.pre.get(prefix, 0)


# Your MapSum object will be instantiated and called as such:
# obj = MapSum()
# obj.insert(key,val)
# param_2 = obj.sum(prefix)

MapSumPrefixDict = MapSum
assert run_ops(MapSumPrefixDict, example_ops, example_args) == example_expected
print("Cách 2 OK")

Cách 2 OK


**Độ phức tạp (Cách 2):** `insert` `O(L^2)` tính cả việc cắt chuỗi `key[:i]` (mỗi lần cắt `O(i)`); với `L <= 50` là rất nhỏ. `sum` `O(L)` (băm `prefix`). Bộ nhớ `O(tổng L^2)` trong trường hợp xấu vì lưu từng tiền tố như một chuỗi riêng.

---

### 7.2. Cách 3 — Trie lưu tổng cây con (delta)

**Giải thích từng đoạn.**

- Mỗi nút có `children` (dict hoặc mảng 26) và `val` = tổng giá trị của mọi key đi qua nút đó = tổng cây con = `S[tiền tố của nút]`.
- `insert(key, val)`: `delta = val - d.get(key, 0)`; đi từ gốc theo từng ký tự (tạo nút nếu thiếu), **cộng `delta` vào `val` của mỗi nút đi qua**.
- `sum(prefix)`: đi theo `prefix`; nếu thiếu nút nào trả `0`; ngược lại trả `val` của nút cuối. Không cần duyệt cây con.

In [11]:
class TrieNode:
    def __init__(self):
        self.children = {}
        self.val = 0

class MapSum:
    def __init__(self):
        self.root = TrieNode()
        self.d = {}

    def insert(self, key: str, val: int) -> None:
        delta = val - self.d.get(key, 0)
        self.d[key] = val
        node = self.root
        for ch in key:
            if ch not in node.children:
                node.children[ch] = TrieNode()
            node = node.children[ch]
            node.val += delta

    def sum(self, prefix: str) -> int:
        node = self.root
        for ch in prefix:
            if ch not in node.children:
                return 0
            node = node.children[ch]
        return node.val

MapSumTrieDelta = MapSum
res = run_ops(MapSumTrieDelta, example_ops, example_args)
print(res)
assert res == example_expected

[None, None, 3, None, 5]


**Code nộp LeetCode (Cách 3):** dùng mảng 26 con như lời giải chuẩn.

In [12]:
class Trie:
    def __init__(self):
        self.children: list = [None] * 26
        self.val: int = 0

    def insert(self, w: str, x: int):
        node = self
        for c in w:
            idx = ord(c) - ord("a")
            if node.children[idx] is None:
                node.children[idx] = Trie()
            node = node.children[idx]
            node.val += x

    def search(self, w: str) -> int:
        node = self
        for c in w:
            idx = ord(c) - ord("a")
            if node.children[idx] is None:
                return 0
            node = node.children[idx]
        return node.val


class MapSum:

    def __init__(self):
        self.d = {}
        self.tree = Trie()

    def insert(self, key: str, val: int) -> None:
        x = val - self.d.get(key, 0)
        self.d[key] = val
        self.tree.insert(key, x)

    def sum(self, prefix: str) -> int:
        return self.tree.search(prefix)


# Your MapSum object will be instantiated and called as such:
# obj = MapSum()
# obj.insert(key,val)
# param_2 = obj.sum(prefix)

MapSumTrie26 = MapSum
assert run_ops(MapSumTrie26, example_ops, example_args) == example_expected
print("Cách 3 OK")

Cách 3 OK


**Độ phức tạp (Cách 3):** `insert` `O(L)`, `sum` `O(L)`; bộ nhớ `O(số nút * 26)` với mảng 26 con (hoặc `O(số nút)` với `dict`); số nút tối đa bằng tổng độ dài các key phân biệt, thực tế nhỏ hơn nhờ chung tiền tố. **Đây là cách nên nộp.**

---

### 7.3. Cách 4 — Trie chỉ lưu giá trị ở nút cuối + duyệt cây con khi `sum`

**Ý tưởng khác.** Không cộng sẵn. Mỗi nút lưu `val` **chỉ nếu một key kết thúc ở đó** (ngược lại `None`/0). `insert` chỉ đặt `val` ở nút cuối (ghi đè là tự nhiên, không cần `delta`!). `sum(prefix)` đi tới nút của `prefix`, rồi **duyệt toàn bộ cây con** (DFS bằng ngăn xếp) cộng các `val`.

**Giải thích từng đoạn.**

- Đi theo `prefix`; thiếu nút -> `0`.
- Ngăn xếp `stack = [node]`; mỗi lần lấy một nút, cộng `val`, đẩy mọi con vào ngăn xếp.
- Ưu: `insert` đơn giản, không cần dict giá trị cũ. Nhược: `sum` phụ thuộc kích thước cây con.

In [13]:
class MapSum:
    def __init__(self):
        self.root = TrieNode()      # dùng lại TrieNode: val chỉ có nghĩa ở nút cuối của một key

    def insert(self, key: str, val: int) -> None:
        node = self.root
        for ch in key:
            if ch not in node.children:
                node.children[ch] = TrieNode()
            node = node.children[ch]
        node.val = val              # ghi đè trực tiếp

    def sum(self, prefix: str) -> int:
        node = self.root
        for ch in prefix:
            if ch not in node.children:
                return 0
            node = node.children[ch]
        total = 0
        stack = [node]
        while stack:
            cur = stack.pop()
            total += cur.val
            stack.extend(cur.children.values())
        return total

MapSumTrieDFS = MapSum
res = run_ops(MapSumTrieDFS, example_ops, example_args)
print(res)
assert res == example_expected

[None, None, 3, None, 5]


**Code nộp LeetCode (Cách 4):**

In [14]:
class MapSum:

    def __init__(self):
        self.root = {}          # nút = dict ký tự -> nút con; khóa "#" lưu giá trị nếu có key kết thúc

    def insert(self, key: str, val: int) -> None:
        node = self.root
        for ch in key:
            node = node.setdefault(ch, {})
        node["#"] = val

    def sum(self, prefix: str) -> int:
        node = self.root
        for ch in prefix:
            if ch not in node:
                return 0
            node = node[ch]
        total = 0
        stack = [node]
        while stack:
            cur = stack.pop()
            for k, v in cur.items():
                if k == "#":
                    total += v
                else:
                    stack.append(v)
        return total


# Your MapSum object will be instantiated and called as such:
# obj = MapSum()
# obj.insert(key,val)
# param_2 = obj.sum(prefix)

MapSumTrieDFS2 = MapSum
assert run_ops(MapSumTrieDFS2, example_ops, example_args) == example_expected
print("Cách 4 OK")

Cách 4 OK


**Độ phức tạp (Cách 4):** `insert` `O(L)`; `sum` `O(L + K)` với `K` là số nút trong cây con của `prefix` (xấu nhất bằng cả cây, tương đương Cách 1). Bộ nhớ `O(số nút)`.

---

### 7.4. Cách 5 — Danh sách key đã sắp xếp + `bisect`

**Ý tưởng.** Mọi key có tiền tố `prefix` nằm **liền nhau** trong danh sách sắp xếp (xem 3.5). Giữ `keys` (đã sắp xếp) và `vals` (dict). `sum(prefix)`: tìm `lo = bisect_left(keys, prefix)`, `hi = bisect_left(keys, prefix + "{")`, cộng `vals[k]` cho `keys[lo:hi]`.

**Giải thích từng đoạn.**

- `insert`: nếu key **chưa có** thì `insort(keys, key)` (chèn giữ thứ tự); luôn `vals[key] = val` (ghi đè).
- `sum`: hai lần nhị phân, rồi cộng đoạn.

In [15]:
class MapSum:
    def __init__(self):
        self.keys = []
        self.vals = {}

    def insert(self, key: str, val: int) -> None:
        if key not in self.vals:
            insort(self.keys, key)
        self.vals[key] = val

    def sum(self, prefix: str) -> int:
        lo = bisect_left(self.keys, prefix)
        hi = bisect_left(self.keys, prefix + "{")      # '{' đứng ngay sau 'z'
        return sum(self.vals[k] for k in self.keys[lo:hi])

MapSumBisect = MapSum
res = run_ops(MapSumBisect, example_ops, example_args)
print(res)
assert res == example_expected

[None, None, 3, None, 5]


**Code nộp LeetCode (Cách 5):**

In [16]:
from bisect import bisect_left, insort

class MapSum:

    def __init__(self):
        self.keys = []
        self.vals = {}

    def insert(self, key: str, val: int) -> None:
        if key not in self.vals:
            insort(self.keys, key)
        self.vals[key] = val

    def sum(self, prefix: str) -> int:
        lo = bisect_left(self.keys, prefix)
        hi = bisect_left(self.keys, prefix + "{")
        return sum(self.vals[k] for k in self.keys[lo:hi])


# Your MapSum object will be instantiated and called as such:
# obj = MapSum()
# obj.insert(key,val)
# param_2 = obj.sum(prefix)

MapSumBisect2 = MapSum
assert run_ops(MapSumBisect2, example_ops, example_args) == example_expected
print("Cách 5 OK")

Cách 5 OK


**Độ phức tạp (Cách 5):** `insert` `O(L log N + N)` (tìm vị trí `O(L log N)`, nhưng chèn vào list Python dịch chuyển phần tử `O(N)`, rất nhanh trong thực tế); `sum` `O(L log N + K * L)` với `K` là số key có tiền tố (cộng đoạn). Bộ nhớ `O(N * L)`.

## 8. TRACE từng bước

### 8.1. Cách 3 trên ví dụ của đề (có ghi đè)

Ta chạy `insert("apple",3)`, `insert("app",2)`, rồi **ghi đè** `insert("apple",10)`, và in "tổng cây con" của từng nút sau mỗi lệnh.

| Lệnh | `delta` | Nút bị cập nhật | `sum("ap")` |
|---|---|---|---|
| `insert("apple", 3)` | 3 - 0 = +3 | a, ap, app, appl, apple đều +3 | 3 |
| `insert("app", 2)` | 2 - 0 = +2 | a, ap, app đều +2 | 5 |
| `insert("apple", 10)` | 10 - 3 = +7 | a, ap, app, appl, apple đều +7 | 12 |

In [17]:
def dump_trie(node, prefix=""):
    for ch in sorted(node.children):
        child = node.children[ch]
        print(f"    {prefix + ch!r:9} tổng cây con = {child.val}")
        dump_trie(child, prefix + ch)

def trace_trie(commands):
    m = MapSumTrieDelta()
    for op, arg in commands:
        if op == "insert":
            key, val = arg
            old = m.d.get(key, 0)
            print(f"insert({key!r}, {val}): giá trị cũ = {old}, delta = {val - old:+d}")
            m.insert(key, val)
            dump_trie(m.root)
        else:
            print(f"sum({arg!r}) = {m.sum(arg)}")
    return m

m = trace_trie([("insert", ("apple", 3)), ("sum", "ap"), ("insert", ("app", 2)), ("sum", "ap"),
                ("insert", ("apple", 10)), ("sum", "ap"), ("sum", "apx"), ("sum", "a")])
assert m.sum("ap") == 12 and m.sum("apx") == 0 and m.sum("a") == 12

insert('apple', 3): giá trị cũ = 0, delta = +3
    'a'       tổng cây con = 3
    'ap'      tổng cây con = 3
    'app'     tổng cây con = 3
    'appl'    tổng cây con = 3
    'apple'   tổng cây con = 3
sum('ap') = 3
insert('app', 2): giá trị cũ = 0, delta = +2
    'a'       tổng cây con = 5
    'ap'      tổng cây con = 5
    'app'     tổng cây con = 5
    'appl'    tổng cây con = 3
    'apple'   tổng cây con = 3
sum('ap') = 5
insert('apple', 10): giá trị cũ = 3, delta = +7
    'a'       tổng cây con = 12
    'ap'      tổng cây con = 12
    'app'     tổng cây con = 12
    'appl'    tổng cây con = 10
    'apple'   tổng cây con = 10
sum('ap') = 12
sum('apx') = 0
sum('a') = 12


### 8.2. Cách 2: bảng `tiền tố -> tổng`

Sau 3 lệnh trên, bảng `pre` chứa `a, ap, app, appl, apple` với tổng lần lượt `12, 12, 12, 10, 10`. Hãy in ra để thấy `app` (key thật, giá trị 2) cộng với `apple` (10) cho 12 ở `app`.

In [18]:
m2 = MapSumPrefixDict()
for key, val in [("apple", 3), ("app", 2), ("apple", 10)]:
    old = m2.d.get(key, 0)
    m2.insert(key, val)
    print(f"insert({key!r}, {val}) delta={val - old:+d} ->", dict(m2.pre))
assert dict(m2.pre) == {"a": 12, "ap": 12, "app": 12, "appl": 10, "apple": 10}

insert('apple', 3) delta=+3 -> {'a': 3, 'ap': 3, 'app': 3, 'appl': 3, 'apple': 3}
insert('app', 2) delta=+2 -> {'a': 5, 'ap': 5, 'app': 5, 'appl': 3, 'apple': 3}
insert('apple', 10) delta=+7 -> {'a': 12, 'ap': 12, 'app': 12, 'appl': 10, 'apple': 10}


### 8.3. Cách 5: hai lần nhị phân tìm đoạn

Với `keys = ["app", "apple", "apt", "bat"]` và `sum("ap")`: ta in `lo`, `hi` và đoạn được cộng.

In [19]:
def trace_bisect(keys, vals, prefix):
    lo = bisect_left(keys, prefix)
    hi = bisect_left(keys, prefix + "{")
    seg = keys[lo:hi]
    print(f"sum({prefix!r}): bisect_left(keys, {prefix!r}) = {lo}; bisect_left(keys, {prefix + '{'!r}) = {hi}; đoạn = {seg}")
    total = sum(vals[k] for k in seg)
    print("  tổng =", total)
    return total

keys = ["app", "apple", "apt", "bat"]
vals = {"app": 2, "apple": 10, "apt": 4, "bat": 7}
assert trace_bisect(keys, vals, "ap") == 16
assert trace_bisect(keys, vals, "app") == 12
assert trace_bisect(keys, vals, "b") == 7
assert trace_bisect(keys, vals, "c") == 0
assert trace_bisect(keys, vals, "") == 23

sum('ap'): bisect_left(keys, 'ap') = 0; bisect_left(keys, 'ap{') = 3; đoạn = ['app', 'apple', 'apt']
  tổng = 16
sum('app'): bisect_left(keys, 'app') = 0; bisect_left(keys, 'app{') = 2; đoạn = ['app', 'apple']
  tổng = 12
sum('b'): bisect_left(keys, 'b') = 3; bisect_left(keys, 'b{') = 4; đoạn = ['bat']
  tổng = 7
sum('c'): bisect_left(keys, 'c') = 4; bisect_left(keys, 'c{') = 4; đoạn = []
  tổng = 0
sum(''): bisect_left(keys, '') = 0; bisect_left(keys, '{') = 4; đoạn = ['app', 'apple', 'apt', 'bat']
  tổng = 23


## 9. Code nộp LeetCode

Ta chọn **Cách 3** (Trie lưu tổng cây con + dict giá trị cũ): `insert` và `sum` đều `O(L)`, ngắn gọn và không phụ thuộc số key.

```python
class Trie:
    def __init__(self):
        self.children: list = [None] * 26
        self.val: int = 0

    def insert(self, w: str, x: int):
        node = self
        for c in w:
            idx = ord(c) - ord("a")
            if node.children[idx] is None:
                node.children[idx] = Trie()
            node = node.children[idx]
            node.val += x

    def search(self, w: str) -> int:
        node = self
        for c in w:
            idx = ord(c) - ord("a")
            if node.children[idx] is None:
                return 0
            node = node.children[idx]
        return node.val


class MapSum:

    def __init__(self):
        self.d = {}
        self.tree = Trie()

    def insert(self, key: str, val: int) -> None:
        x = val - self.d.get(key, 0)
        self.d[key] = val
        self.tree.insert(key, x)

    def sum(self, prefix: str) -> int:
        return self.tree.search(prefix)
```

Cell dưới định nghĩa lại đúng bản này để chạy qua bộ `run_ops`.

In [20]:
class Trie:
    def __init__(self):
        self.children: list = [None] * 26
        self.val: int = 0

    def insert(self, w: str, x: int):
        node = self
        for c in w:
            idx = ord(c) - ord("a")
            if node.children[idx] is None:
                node.children[idx] = Trie()
            node = node.children[idx]
            node.val += x

    def search(self, w: str) -> int:
        node = self
        for c in w:
            idx = ord(c) - ord("a")
            if node.children[idx] is None:
                return 0
            node = node.children[idx]
        return node.val


class MapSum:

    def __init__(self):
        self.d = {}
        self.tree = Trie()

    def insert(self, key: str, val: int) -> None:
        x = val - self.d.get(key, 0)
        self.d[key] = val
        self.tree.insert(key, x)

    def sum(self, prefix: str) -> int:
        return self.tree.search(prefix)

MapSumFinal = MapSum
res = run_ops(MapSumFinal, example_ops, example_args)
print(res)
assert res == example_expected

[None, None, 3, None, 5]


## 10. Vì sao đúng

**Bất biến (Cách 2 và 3).** Sau mỗi `insert`, với **mọi** tiền tố `p`: `S[p] = tổng val hiện tại của mọi key bắt đầu bằng p`.

- *Khởi tạo:* map rỗng, mọi `S[p] = 0`. Đúng.
- *Duy trì:* xét `insert(key, val)` với giá trị cũ `old` (0 nếu key mới). Chỉ giá trị của `key` đổi, từ `old` thành `val`. Một tiền tố `p` có tổng bị ảnh hưởng **khi và chỉ khi** `key` bắt đầu bằng `p`, tức `p` là một trong `len(key)` tiền tố của `key` (hoặc rỗng). Với các tiền tố đó, tổng đổi đúng `val - old = delta`; với các tiền tố khác tổng giữ nguyên. Vòng lặp cộng `delta` vào chính các tiền tố đó. Bất biến giữ.
- *Trả lời:* `sum(prefix) = S[prefix]` (0 nếu chưa có nút/mục tương ứng, vì khi đó không key nào có tiền tố đó).

Đối với Trie, "nút ứng với tiền tố `p`" chính là nút ở cuối đường đi `p`, nên cập nhật mọi nút trên đường đi của `key` = cập nhật đúng các tiền tố của `key`.

**Cách 4.** Nút cuối của `prefix` có cây con chứa đúng các key bắt đầu bằng `prefix`; mỗi key kết thúc ở đúng một nút và nút đó lưu giá trị mới nhất (ghi đè). Cộng mọi `val` trong cây con ra đáp án.

**Cách 5.** Nếu `key` bắt đầu bằng `prefix` thì `prefix <= key` và `key < prefix + "{"` (vì ký tự sau `prefix` trong `key`, là chữ thường, nhỏ hơn `"{"`; hoặc `key == prefix`). Ngược lại một key nằm trong `[prefix, prefix + "{")` mà không bắt đầu bằng `prefix` là không thể: nó phải khớp `prefix` tới hết độ dài `prefix`. Do đó đoạn `[lo, hi)` chính xác là các key cần tìm.

**Kiểm chứng bằng máy:** so sánh 5 cách với một oracle viết bằng cách khác hẳn (phát lại toàn bộ lịch sử `insert` để tìm giá trị cuối cùng của từng key, rồi quét).

In [21]:
class Oracle:
    """Oracle độc lập: lưu mọi lần insert theo thứ tự, mỗi lần sum phát lại lịch sử."""
    def __init__(self):
        self.history = []

    def insert(self, key, val):
        self.history.append((key, val))

    def sum(self, prefix):
        final = {}
        for k, v in self.history:         # lần sau ghi đè lần trước
            final[k] = v
        total = 0
        for k, v in final.items():
            if len(k) >= len(prefix) and k[:len(prefix)] == prefix:
                total += v
        return total

assert run_ops(Oracle, ["Oracle"] + example_ops[1:], example_args) == example_expected

implementations = {
    "scan": MapSumScan,
    "prefix_dict": MapSumPrefixDict,
    "trie_delta": MapSumTrieDelta,
    "trie26": MapSumTrie26,
    "trie_dfs": MapSumTrieDFS,
    "trie_dfs_dict": MapSumTrieDFS2,
    "bisect": MapSumBisect,
    "final": MapSumFinal,
}

def gen_ops(rng, n_ops, alphabet, max_len, max_val, p_insert=0.5):
    ops, args = ["MapSum"], [[]]
    for _ in range(n_ops):
        L = rng.randint(1, max_len)
        s = "".join(rng.choice(alphabet) for _ in range(L))
        if rng.random() < p_insert:
            ops.append("insert"); args.append([s, rng.randint(1, max_val)])
        else:
            ops.append("sum"); args.append([s])
    return ops, args

rng = random.Random(677)
for _ in range(300):
    ops, args = gen_ops(rng, 50, "ab", 4, 1000)
    ref = run_ops(Oracle, ["Oracle"] + ops[1:], args)
    for name, cls in implementations.items():
        assert run_ops(cls, ops, args) == ref, (name, ops, args)
print("5 cách (8 bản cài đặt) khớp oracle trên 300 chuỗi lệnh ngẫu nhiên: OK")

5 cách (8 bản cài đặt) khớp oracle trên 300 chuỗi lệnh ngẫu nhiên: OK

## 11. Time / Space Complexity

Gọi `N` là số key phân biệt, `L` là độ dài tối đa của key/prefix (`<= 50`), `K` là số key khớp một tiền tố, `V` là số nút của Trie.

| Cách | `insert` | `sum` | Space |
|---|---|---|---|
| 1. Quét dict | `O(L)` | `O(N * L)` | `O(N * L)` |
| 2. Bảng băm tiền tố | `O(L^2)` | `O(L)` | `O(N * L^2)` |
| 3. Trie lưu tổng | `O(L)` | `O(L)` | `O(V)` (mảng 26: `O(26 V)`) |
| 4. Trie + DFS cây con | `O(L)` | `O(L + K')` (`K'` nút cây con) | `O(V)` |
| 5. Danh sách sắp xếp + bisect | `O(L log N + N)` | `O(L log N + K * L)` | `O(N * L)` |

Với đề (`<= 50` lệnh) mọi cách đều tức thì. Để thấy khác biệt, ta đo với dữ liệu lớn hơn nhiều so với ràng buộc: 10.000 key, 300 truy vấn `sum`.

In [22]:
rng = random.Random(11)
alphabet = "abc"
big_keys = ["".join(rng.choice(alphabet) for _ in range(rng.randint(5, 50))) for _ in range(10000)]
big_prefixes = ["".join(rng.choice(alphabet) for _ in range(rng.randint(1, 6))) for _ in range(300)]

ref = None
for name in ["scan", "prefix_dict", "trie_delta", "trie_dfs", "bisect"]:
    cls = implementations[name]
    t0 = time.perf_counter()
    obj = cls()
    for i, k in enumerate(big_keys):
        obj.insert(k, i % 1000 + 1)
    t1 = time.perf_counter()
    answers = [obj.sum(p) for p in big_prefixes]
    t2 = time.perf_counter()
    if ref is None:
        ref = answers
    assert answers == ref, name
    print(f"{name:12} insert 10000 key: {1000 * (t1 - t0):8.1f} ms | 300 lần sum: {1000 * (t2 - t1):9.1f} ms")

scan         insert 10000 key:      2.9 ms | 300 lần sum:     141.8 ms
prefix_dict  insert 10000 key:    101.8 ms | 300 lần sum:       0.1 ms


trie_delta   insert 10000 key:    250.1 ms | 300 lần sum:       0.2 ms


trie_dfs     insert 10000 key:    244.2 ms | 300 lần sum:     823.1 ms
bisect       insert 10000 key:     47.6 ms | 300 lần sum:      23.7 ms


## 12. Bộ test

Gồm: ví dụ của đề (qua `run_ops`), edge case, phản ví dụ cho các cách sai, chuỗi lệnh ngẫu nhiên có seed cố định (đúng ràng buộc, nhiều ghi đè), và chạy nhiều đối tượng độc lập.

In [23]:
def check_all(ops, args, expected=None):
    ref = run_ops(Oracle, ["Oracle"] + ops[1:], args)
    if expected is not None:
        assert ref == expected, ("oracle khác kỳ vọng", ref, expected)
    for name, cls in implementations.items():
        assert run_ops(cls, ops, args) == ref, (name, ops, args)
    return ref

# 12.1 ví dụ đề
check_all(example_ops, example_args, example_expected)
print("12.1 ví dụ đề: OK")

# 12.2 edge case
cases = {
    "sum khi map rỗng": (["MapSum", "sum"], [[], ["a"]], [None, 0]),
    "prefix không tồn tại": (["MapSum", "insert", "sum"], [[], ["apple", 5], ["b"]], [None, None, 0]),
    "prefix dài hơn mọi key": (["MapSum", "insert", "sum"], [[], ["app", 5], ["apple"]], [None, None, 0]),
    "prefix = key": (["MapSum", "insert", "sum"], [[], ["app", 5], ["app"]], [None, None, 5]),
    "ghi đè giảm giá trị": (["MapSum", "insert", "insert", "sum"], [[], ["a", 10], ["a", 3], ["a"]], [None, None, None, 3]),
    "ghi đè nhiều lần": (["MapSum", "insert", "insert", "insert", "sum", "sum"],
                         [[], ["ab", 1], ["ab", 1000], ["ab", 7], ["a"], ["ab"]], [None, None, None, None, 7, 7]),
    "nhiều key chung tiền tố": (["MapSum", "insert", "insert", "insert", "sum", "sum"],
                                [[], ["a", 1], ["ab", 2], ["abc", 4], ["a"], ["ab"]], [None, None, None, None, 7, 6]),
    "key một ký tự": (["MapSum", "insert", "insert", "sum", "sum"], [[], ["a", 1], ["b", 2], ["a"], ["b"]], [None, None, None, 1, 2]),
    "key dài 50": (["MapSum", "insert", "sum", "sum"], [[], ["z" * 50, 1000], ["z" * 25], ["z" * 50]], [None, None, 1000, 1000]),
    "'z' và chốt chặn": (["MapSum", "insert", "insert", "sum"], [[], ["apzz", 8], ["ap", 1], ["ap"]], [None, None, None, 9]),
}
for name, (ops, args, expected) in cases.items():
    check_all(ops, args, expected)
print("12.2 edge case: OK", len(cases), "ca")

# 12.3 phản ví dụ: các cài đặt sai bị lộ
class WrongNoDelta:                 # quên trừ giá trị cũ khi ghi đè
    def __init__(self):
        self.root = TrieNode()
    def insert(self, key, val):
        node = self.root
        for ch in key:
            node = node.children.setdefault(ch, TrieNode())
            node.val += val
    def sum(self, prefix):
        node = self.root
        for ch in prefix:
            if ch not in node.children:
                return 0
            node = node.children[ch]
        return node.val

class WrongSubstring:               # dùng "in" thay vì startswith
    def __init__(self):
        self.d = {}
    def insert(self, key, val):
        self.d[key] = val
    def sum(self, prefix):
        return sum(v for k, v in self.d.items() if prefix in k)

class WrongSentinelZ:               # chốt chặn bằng 'z' thay vì '{'
    def __init__(self):
        self.keys, self.vals = [], {}
    def insert(self, key, val):
        if key not in self.vals:
            insort(self.keys, key)
        self.vals[key] = val
    def sum(self, prefix):
        lo = bisect_left(self.keys, prefix)
        hi = bisect_left(self.keys, prefix + "z")
        return sum(self.vals[k] for k in self.keys[lo:hi])

r1 = run_ops(WrongNoDelta, ["WrongNoDelta", "insert", "insert", "sum"], [[], ["a", 3], ["a", 5], ["a"]])
r2 = run_ops(WrongSubstring, ["WrongSubstring", "insert", "sum"], [[], ["banana", 4], ["an"]])
r3 = run_ops(WrongSentinelZ, ["WrongSentinelZ", "insert", "insert", "sum"], [[], ["apzz", 8], ["ap", 1], ["ap"]])
print("quên delta:", r1[-1], "(đúng 5) | in thay startswith:", r2[-1], "(đúng 0) | chốt chặn 'z':", r3[-1], "(đúng 9)")
assert r1[-1] == 8 and r2[-1] == 4 and r3[-1] == 1
print("12.3 phản ví dụ: OK (cả ba cách sai đều bị lộ)")

# 12.4 ngẫu nhiên có seed: nhiều kích thước và bảng chữ cái, đúng ràng buộc
rng = random.Random(2077)
for n_ops, alpha, max_len in [(50, "ab", 3), (50, "abc", 6), (50, "abcdefghijklmnopqrstuvwxyz", 50), (400, "ab", 5), (800, "abc", 12)]:
    for _ in range(15):
        ops, args = gen_ops(rng, n_ops, alpha, max_len, 1000, p_insert=rng.choice([0.2, 0.5, 0.8]))
        check_all(ops, args)
print("12.4 ngẫu nhiên có seed: OK")

# 12.5 nhiều đối tượng độc lập (kiểm tra không dùng chung trạng thái)
for name, cls in implementations.items():
    a, b = cls(), cls()
    a.insert("ab", 5)
    b.insert("ab", 7)
    b.insert("ac", 1)
    assert a.sum("a") == 5 and b.sum("a") == 8, name
print("12.5 nhiều đối tượng độc lập: OK")

12.1 ví dụ đề: OK
12.2 edge case: OK 10 ca
quên delta: 8 (đúng 5) | in thay startswith: 4 (đúng 0) | chốt chặn 'z': 1 (đúng 9)
12.3 phản ví dụ: OK (cả ba cách sai đều bị lộ)


12.4 ngẫu nhiên có seed: OK
12.5 nhiều đối tượng độc lập: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên trừ giá trị cũ khi ghi đè

Cộng `val` vào mọi nút mà không trừ giá trị cũ của key. Ghi đè `a: 3 -> 5` cho tổng `3 + 5 = 8` thay vì `5`.

In [24]:
res_wrong = run_ops(WrongNoDelta, ["WrongNoDelta", "insert", "insert", "sum"], [[], ["a", 3], ["a", 5], ["a"]])
res_right = run_ops(MapSumFinal, ["MapSum", "insert", "insert", "sum"], [[], ["a", 3], ["a", 5], ["a"]])
print("sai :", res_wrong)
print("đúng:", res_right)
assert res_wrong[-1] == 8 and res_right[-1] == 5

sai : [None, None, None, 8]
đúng: [None, None, None, 5]


Vì sao sai: đề nói key đã tồn tại thì giá trị cũ bị **ghi đè**, không phải cộng dồn. **Đúng:** giữ `d[key]` và cộng `delta = val - d.get(key, 0)`.

### Lỗi 2 — `prefix in key` thay vì `key.startswith(prefix)`

`in` kiểm tra tồn tại ở **mọi vị trí**: `"an" in "banana"` đúng, nhưng `banana` không bắt đầu bằng `an`.

In [25]:
res_wrong = run_ops(WrongSubstring, ["WrongSubstring", "insert", "sum"], [[], ["banana", 4], ["an"]])
res_right = run_ops(MapSumScan, ["MapSum", "insert", "sum"], [[], ["banana", 4], ["an"]])
print("sai :", res_wrong, "| đúng:", res_right)
assert res_wrong[-1] == 4 and res_right[-1] == 0

sai : [None, None, 4] | đúng: [None, None, 0]


Vì sao sai: tiền tố là phần **đầu** chuỗi. **Đúng:** `key.startswith(prefix)`.

### Lỗi 3 — Chốt chặn `bisect` sai (`prefix + "z"`)

Trong Cách 5, dùng `prefix + "z"` làm cận trên bỏ sót các key như `apzz` (vì `"apzz" >= "apz"`).

In [26]:
res_wrong = run_ops(WrongSentinelZ, ["WrongSentinelZ", "insert", "insert", "sum"], [[], ["apzz", 8], ["ap", 1], ["ap"]])
res_right = run_ops(MapSumBisect, ["MapSum", "insert", "insert", "sum"], [[], ["apzz", 8], ["ap", 1], ["ap"]])
print("sai :", res_wrong, "| đúng:", res_right)
assert res_wrong[-1] == 1 and res_right[-1] == 9

sai : [None, None, None, 1] | đúng: [None, None, None, 9]


Vì sao sai: cận trên phải **lớn hơn mọi chuỗi** bắt đầu bằng `prefix`. Ký tự `'{'` (mã 123) đứng ngay sau `'z'` (mã 122) nên `prefix + '{'` thỏa. **Đúng:** `bisect_left(keys, prefix + '{')`.

### Lỗi 4 — Trạng thái đặt ở cấp lớp (dùng chung giữa các đối tượng)

Khi viết `d = {}` ngay trong thân lớp (không phải trong `__init__`), mọi đối tượng chia sẻ cùng một dict. LeetCode tạo nhiều đối tượng qua các test case nên kết quả bị nhiễm.

In [27]:
class MapSumShared:
    d = {}                                   # SAI: thuộc về lớp, dùng chung
    def insert(self, key, val):
        self.d[key] = val
    def sum(self, prefix):
        return sum(v for k, v in self.d.items() if k.startswith(prefix))

x, y = MapSumShared(), MapSumShared()
x.insert("apple", 3)
print("y chưa insert gì nhưng y.sum('ap') =", y.sum("ap"), "(đáng ra là 0)")
assert y.sum("ap") == 3

p, q = MapSumScan(), MapSumScan()
p.insert("apple", 3)
print("đúng: q.sum('ap') =", q.sum("ap"))
assert q.sum("ap") == 0

y chưa insert gì nhưng y.sum('ap') = 3 (đáng ra là 0)
đúng: q.sum('ap') = 0


Vì sao sai: thuộc tính của lớp là một đối tượng duy nhất. **Đúng:** khởi tạo trong `__init__` bằng `self.d = {}`.

### Lỗi 5 — Truy cập nút `None` / thiếu nút khi `sum`

Trong Trie, đi theo `prefix` mà không kiểm tra con có tồn tại sẽ gây lỗi (`AttributeError` với `None`, hoặc `KeyError`).

In [28]:
def wrong_search(root, prefix):
    node = root
    for ch in prefix:
        node = node.children[ch]            # KeyError nếu thiếu
    return node.val

m = MapSumTrieDelta()
m.insert("apple", 3)
try:
    wrong_search(m.root, "bz")
except KeyError as ex:
    print("lỗi KeyError với prefix không tồn tại:", ex)
print("đúng:", m.sum("bz"))
assert m.sum("bz") == 0

lỗi KeyError với prefix không tồn tại: 'b'
đúng: 0


Vì sao sai: prefix không có trong cây nghĩa là không key nào khớp; đáp án phải là `0`. **Đúng:** `if ch not in node.children: return 0`.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu Trie:** "tiền tố", "bắt đầu bằng", nhiều truy vấn trên một tập chuỗi.
- **Dấu hiệu "cộng sẵn":** truy vấn lặp lại trên cùng dữ liệu, cập nhật ít -> lưu kết quả tổng hợp ở nút/khóa, cập nhật bằng **delta** thay vì tính lại.
- **Ghi đè = trừ cái cũ, cộng cái mới.** Luôn giữ một dict "giá trị hiện tại" để biết `old`.
- **Nhiều lần cập nhật, ít truy vấn** -> Cách 4 (chỉ ghi nút cuối) đơn giản hơn; **ít cập nhật, nhiều truy vấn** -> Cách 3.
- **Giới hạn nhỏ (50 lệnh)** cho phép cả cách quét; nhưng trong phỏng vấn người ta mong thấy Trie.
- **Bài thiết kế:** trạng thái trong `__init__`, không đặt ở cấp lớp; tự dựng bộ chạy lệnh để thử chuỗi lệnh dài.
- Chốt chặn `'{'` (hoặc `chr(ord('z') + 1)`) cho "cận trên của mọi chuỗi chữ thường bắt đầu bằng prefix".
- Mảng 26 con nhanh và gọn khi bảng chữ cái cố định; dict con linh hoạt hơn khi ký tự bất kỳ.

## 15. Biến thể

1. **Xóa key** (`delete(key)`): trừ `val` hiện tại của key khỏi mọi tiền tố, rồi xóa khỏi dict.
2. **Đếm số key có tiền tố** (giống `startsWith` của Implement Trie 208): lưu `cnt` thay vì tổng.
3. **Tổng theo hậu tố** (suffix): dùng Trie trên chuỗi đảo ngược.
4. Bài liên quan: Implement Trie (208), Design Add and Search Words (211), Map Sum with delete trong phỏng vấn, Implement Magic Dictionary (676), Prefix and Suffix Search (745).

Dưới đây là code cho biến thể 1 và 2, kiểm bằng oracle.

In [29]:
class MapSumPlus:
    """insert / delete / sum / count_prefix cùng lúc, mỗi nút giữ (tổng, số key)."""
    def __init__(self):
        self.root = TrieNode()
        self.d = {}

    def _walk(self, key, dsum, dcnt):
        node = self.root
        for ch in key:
            node = node.children.setdefault(ch, TrieNode())
            node.val += dsum
            node.cnt = getattr(node, "cnt", 0) + dcnt

    def insert(self, key, val):
        old = self.d.get(key)
        self._walk(key, val - (old or 0), 0 if old is not None else 1)
        self.d[key] = val

    def delete(self, key):
        if key not in self.d:
            return False
        self._walk(key, -self.d.pop(key), -1)
        return True

    def _find(self, prefix):
        node = self.root
        for ch in prefix:
            if ch not in node.children:
                return None
            node = node.children[ch]
        return node

    def sum(self, prefix):
        node = self._find(prefix)
        return node.val if node else 0

    def count_prefix(self, prefix):
        node = self._find(prefix)
        return getattr(node, "cnt", 0) if node else 0

mp = MapSumPlus()
mp.insert("apple", 3); mp.insert("app", 2); mp.insert("apple", 10)
print(mp.sum("ap"), mp.count_prefix("ap"))          # 12 2
print(mp.delete("apple"), mp.sum("ap"), mp.count_prefix("ap"))   # True 2 1
print(mp.delete("zzz"))                                          # False
assert (mp.sum("ap"), mp.count_prefix("ap"), mp.count_prefix("a")) == (2, 1, 1)

rng = random.Random(15)
for _ in range(200):
    mp, ref = MapSumPlus(), {}
    for _ in range(60):
        k = "".join(rng.choice("ab") for _ in range(rng.randint(1, 4)))
        r = rng.random()
        if r < 0.4:
            v = rng.randint(1, 1000); mp.insert(k, v); ref[k] = v
        elif r < 0.55:
            assert mp.delete(k) == (k in ref); ref.pop(k, None)
        else:
            assert mp.sum(k) == sum(v for kk, v in ref.items() if kk.startswith(k))
            assert mp.count_prefix(k) == sum(1 for kk in ref if kk.startswith(k))
print("Biến thể delete/count: OK")

12 2
True 2 1
False
Biến thể delete/count: OK


## 16. So sánh các cách

| Cách | Ý tưởng | `insert` | `sum` | Space | Khi nào dùng |
|---|---|---|---|---|---|
| 1. Quét dict | `startswith` trên mọi key | `O(L)` | `O(N L)` | `O(N L)` | Dữ liệu rất nhỏ; làm đối chứng |
| 2. Bảng băm tiền tố | `tiền tố -> tổng`, delta | `O(L^2)` | `O(L)` | `O(N L^2)` | Dễ viết; `L` nhỏ |
| 3. Trie lưu tổng | Tổng cây con, delta | `O(L)` | `O(L)` | `O(V)` | **Cách nộp chính**: truy vấn nhiều |
| 4. Trie + DFS | Giá trị ở nút cuối, duyệt cây con | `O(L)` | `O(L + K')` | `O(V)` | Cập nhật nhiều, truy vấn ít |
| 5. Sorted + bisect | Đoạn liên tiếp trong danh sách | `O(L log N + N)` | `O(L log N + K L)` | `O(N L)` | Khi cần duyệt/trích key theo thứ tự |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output (list kết quả) của chuỗi lệnh sau, rồi chạy `run_ops` để kiểm tra:

```python
ops  = ["MapSum", "insert", "insert", "sum", "insert", "sum", "sum"]
args = [[], ["ab", 4], ["abc", 1], ["ab"], ["ab", 10], ["a"], ["abcd"]]
```

In [30]:
# Bài 1: ghi dự đoán của bạn (chưa chạy code mẫu)
my_guess = None      # ví dụ [None, None, ...]

### Bài 2 — Tính tay Trie (dễ-vừa)

Chạy: `insert("ab", 5)`, `insert("a", 2)`, `insert("ab", 1)`. Hãy ghi "tổng cây con" của các nút `a` và `ab` sau mỗi lệnh (3 cặp), và giá trị `sum("a")`, `sum("ab")`, `sum("b")` cuối cùng.

In [31]:
# Bài 2: điền kết quả tính tay
hand_nodes = None         # list 3 phần tử, mỗi phần tử (tổng nút a, tổng nút ab) sau từng lệnh
hand_sums = None          # (sum("a"), sum("ab"), sum("b"))

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành `MapSum` bằng Trie dict lưu tổng cây con.

In [32]:
class ExerciseMapSum3:
    def __init__(self):
        self.d = {}
        self.root = {"sum": 0, "next": {}}

    def insert(self, key: str, val: int) -> None:
        # TODO 1: delta = val - giá trị cũ của key (0 nếu key mới)
        # TODO 2: cập nhật self.d[key]
        # TODO 3: đi từ gốc theo từng ký tự (tạo nút {"sum": 0, "next": {}} nếu thiếu) và cộng delta vào nút
        pass

    def sum(self, prefix: str) -> int:
        # TODO 4: đi theo prefix, thiếu nút thì trả 0, ngược lại trả node["sum"]
        return None


# assert run_ops(ExerciseMapSum3, ["ExerciseMapSum3"] + example_ops[1:], example_args) == example_expected

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 5, hãy tự viết `ExerciseMapSum4` dùng **danh sách đã sắp xếp + bisect** (chú ý ghi đè và chốt chặn).

In [33]:
class ExerciseMapSum4:
    # TODO
    pass


# assert run_ops(ExerciseMapSum4, ["ExerciseMapSum4"] + example_ops[1:], example_args) == example_expected

### Bài 5 — Mini challenge: `MapSum` có `delete` và `top(prefix)` (khó)

Mở rộng: thêm `delete(key) -> bool` (xóa key, trả `True` nếu có) và `count(prefix) -> int` (số key hiện có bắt đầu bằng `prefix`). `insert`, `sum` giữ nguyên ngữ nghĩa. Ví dụ:

```python
m = ExerciseMapSum5()
m.insert("apple", 3); m.insert("app", 2); m.insert("apple", 10)
m.sum("ap")       -> 12
m.count("ap")     -> 2
m.delete("apple") -> True
m.sum("ap")       -> 2
m.count("ap")     -> 1
m.delete("apple") -> False
```
Gợi ý: Trie, mỗi nút giữ cả `sum` lẫn `cnt`; xóa = cập nhật delta âm.

In [34]:
class ExerciseMapSum5:
    # TODO: các method insert, sum, delete, count
    pass


# m = ExerciseMapSum5()
# m.insert("apple", 3); m.insert("app", 2); m.insert("apple", 10)
# assert (m.sum("ap"), m.count("ap")) == (12, 2)
# assert m.delete("apple") is True
# assert (m.sum("ap"), m.count("ap")) == (2, 1)
# assert m.delete("apple") is False

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: theo dõi map sau từng lệnh; `insert` cùng key thì ghi đè.
- Mức 2: sau `insert("ab", 10)`, map = `{ab: 10, abc: 1}`.
- Mức 3: kết quả `[None, None, None, 5, None, 11, 0]` (`sum("ab") = 4 + 1 = 5`, sau ghi đè `sum("a") = 10 + 1 = 11`, `sum("abcd") = 0`).

**Bài 2**
- Mức 1: lệnh 1: `ab = 5` -> nút `a` = 5, nút `ab` = 5.
- Mức 2: lệnh 2: `a = 2` -> chỉ nút `a` +2 = 7. Lệnh 3: `ab` 5 -> 1, delta = -4.
- Mức 3: `[(5, 5), (7, 5), (3, 1)]` và `sum` cuối: `(3, 1, 0)`.

**Bài 3**
- Mức 1: `delta = val - self.d.get(key, 0)`.
- Mức 2: duyệt `node = self.root; for ch in key: node = node["next"].setdefault(ch, {"sum": 0, "next": {}}); node["sum"] += delta`.
- Mức 3: `sum`: `for ch in prefix: if ch not in node["next"]: return 0; node = node["next"][ch]`; trả `node["sum"]`.

**Bài 4**
- Mức 1: `self.keys` (list) và `self.vals` (dict).
- Mức 2: chỉ `insort` khi key chưa có.
- Mức 3: `lo = bisect_left(keys, prefix)`, `hi = bisect_left(keys, prefix + "{")`, `return sum(vals[k] for k in keys[lo:hi])`.

**Bài 5**
- Mức 1: nút có `sum` và `cnt`.
- Mức 2: `insert` key mới: `cnt` +1 trên đường đi; key đã có: `cnt` giữ nguyên, `sum` + delta.
- Mức 3: `delete`: nếu không có key trả `False`; ngược lại cộng `-val` vào `sum` và `-1` vào `cnt` trên đường đi, `pop` khỏi dict, trả `True`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [35]:
ops1 = ["MapSum", "insert", "insert", "sum", "insert", "sum", "sum"]
args1 = [[], ["ab", 4], ["abc", 1], ["ab"], ["ab", 10], ["a"], ["abcd"]]
out1 = run_ops(MapSumFinal, ops1, args1)
print(out1)
assert out1 == [None, None, None, 5, None, 11, 0]
assert run_ops(Oracle, ["Oracle"] + ops1[1:], args1) == out1

[None, None, None, 5, None, 11, 0]


### Đáp án Bài 2

| Lệnh | nút `a` | nút `ab` |
|---|---|---|
| `insert("ab", 5)` | 5 | 5 |
| `insert("a", 2)` | 7 | 5 |
| `insert("ab", 1)` | 3 | 1 |

`sum("a") = 3`, `sum("ab") = 1`, `sum("b") = 0`.

In [36]:
m = MapSumTrieDelta()
states = []
for k, v in [("ab", 5), ("a", 2), ("ab", 1)]:
    m.insert(k, v)
    states.append((m.root.children["a"].val, m.root.children["a"].children["b"].val))
hand_nodes_solution = [(5, 5), (7, 5), (3, 1)]
hand_sums_solution = (3, 1, 0)
assert states == hand_nodes_solution
assert (m.sum("a"), m.sum("ab"), m.sum("b")) == hand_sums_solution
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [37]:
class ExerciseMapSum3Solution:
    def __init__(self):
        self.d = {}
        self.root = {"sum": 0, "next": {}}

    def insert(self, key: str, val: int) -> None:
        delta = val - self.d.get(key, 0)
        self.d[key] = val
        node = self.root
        for ch in key:
            node = node["next"].setdefault(ch, {"sum": 0, "next": {}})
            node["sum"] += delta

    def sum(self, prefix: str) -> int:
        node = self.root
        for ch in prefix:
            if ch not in node["next"]:
                return 0
            node = node["next"][ch]
        return node["sum"]

assert run_ops(ExerciseMapSum3Solution, ["ExerciseMapSum3Solution"] + example_ops[1:], example_args) == example_expected
rng = random.Random(33)
for _ in range(100):
    ops, args = gen_ops(rng, 50, "ab", 4, 1000)
    ops = ["ExerciseMapSum3Solution"] + ops[1:]
    assert run_ops(ExerciseMapSum3Solution, ops, args) == run_ops(Oracle, ["Oracle"] + ops[1:], args)
print("Bài 3 OK")

Bài 3 OK

### Đáp án Bài 4

In [38]:
class ExerciseMapSum4Solution:
    def __init__(self):
        self.keys = []
        self.vals = {}

    def insert(self, key: str, val: int) -> None:
        if key not in self.vals:
            insort(self.keys, key)
        self.vals[key] = val

    def sum(self, prefix: str) -> int:
        lo = bisect_left(self.keys, prefix)
        hi = bisect_left(self.keys, prefix + "{")
        return sum(self.vals[k] for k in self.keys[lo:hi])

assert run_ops(ExerciseMapSum4Solution, ["ExerciseMapSum4Solution"] + example_ops[1:], example_args) == example_expected
rng = random.Random(44)
for _ in range(100):
    ops, args = gen_ops(rng, 50, "abz", 4, 1000)
    ops = ["ExerciseMapSum4Solution"] + ops[1:]
    assert run_ops(ExerciseMapSum4Solution, ops, args) == run_ops(Oracle, ["Oracle"] + ops[1:], args)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [39]:
class ExerciseMapSum5Solution:
    def __init__(self):
        self.d = {}
        self.root = {"sum": 0, "cnt": 0, "next": {}}

    def _walk(self, key, dsum, dcnt):
        node = self.root
        for ch in key:
            node = node["next"].setdefault(ch, {"sum": 0, "cnt": 0, "next": {}})
            node["sum"] += dsum
            node["cnt"] += dcnt

    def insert(self, key: str, val: int) -> None:
        if key in self.d:
            self._walk(key, val - self.d[key], 0)
        else:
            self._walk(key, val, 1)
        self.d[key] = val

    def delete(self, key: str) -> bool:
        if key not in self.d:
            return False
        self._walk(key, -self.d.pop(key), -1)
        return True

    def _find(self, prefix):
        node = self.root
        for ch in prefix:
            if ch not in node["next"]:
                return None
            node = node["next"][ch]
        return node

    def sum(self, prefix: str) -> int:
        node = self._find(prefix)
        return node["sum"] if node else 0

    def count(self, prefix: str) -> int:
        node = self._find(prefix)
        return node["cnt"] if node else 0

m = ExerciseMapSum5Solution()
m.insert("apple", 3); m.insert("app", 2); m.insert("apple", 10)
assert (m.sum("ap"), m.count("ap")) == (12, 2)
assert m.delete("apple") is True
assert (m.sum("ap"), m.count("ap")) == (2, 1)
assert m.delete("apple") is False

rng = random.Random(55)
for _ in range(200):
    m, ref = ExerciseMapSum5Solution(), {}
    for _ in range(60):
        k = "".join(rng.choice("ab") for _ in range(rng.randint(1, 4)))
        r = rng.random()
        if r < 0.4:
            v = rng.randint(1, 1000); m.insert(k, v); ref[k] = v
        elif r < 0.55:
            assert m.delete(k) == (k in ref); ref.pop(k, None)
        else:
            assert m.sum(k) == sum(v for kk, v in ref.items() if kk.startswith(k))
            assert m.count(k) == sum(1 for kk in ref if kk.startswith(k))
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (Trie lưu tổng cây con + delta)

```python
class MapSum:

    def __init__(self):
        self.d = {}                        # key -> giá trị hiện tại
        self.root = {}                     # nút: {ký tự: nút con}, khóa "#" giữ tổng cây con

    def insert(self, key: str, val: int) -> None:
        delta = val - self.d.get(key, 0)   # ghi đè = trừ cũ cộng mới
        self.d[key] = val
        node = self.root
        for ch in key:
            node = node.setdefault(ch, {"#": 0})
            node["#"] += delta

    def sum(self, prefix: str) -> int:
        node = self.root
        for ch in prefix:
            if ch not in node:
                return 0
            node = node[ch]
        return node["#"]
```

(Biến thể gọn dùng dict chung cho cả "con" và "tổng": khóa `"#"` không trùng với chữ thường. Nút gốc không có khóa `"#"` nhưng không bao giờ bị đọc vì prefix có ít nhất 1 ký tự theo đề.)

### Checklist trước khi submit

- [ ] Trạng thái nằm trong `__init__` (`self.`), không đặt ở cấp lớp?
- [ ] `insert` xử lý ghi đè: `delta = val - giá trị cũ`, và cập nhật dict giá trị?
- [ ] `sum` với prefix không tồn tại trả `0` (không lỗi, không tạo nút mới)?
- [ ] Tiền tố dùng `startswith`/đi theo cây, không phải `in`?
- [ ] Nếu dùng bisect: chốt chặn `prefix + "{"`, không phải `"z"`; chỉ `insort` khi key mới?
- [ ] Tên lớp/method đúng: `MapSum`, `insert(key, val)`, `sum(prefix)`; `insert` không trả giá trị?
- [ ] Đã thử: map rỗng, prefix = key, prefix dài hơn key, ghi đè giảm giá trị?

In [40]:
# Kiểm tra template mục 20
class MapSum:

    def __init__(self):
        self.d = {}
        self.root = {}

    def insert(self, key: str, val: int) -> None:
        delta = val - self.d.get(key, 0)
        self.d[key] = val
        node = self.root
        for ch in key:
            node = node.setdefault(ch, {"#": 0})
            node["#"] += delta

    def sum(self, prefix: str) -> int:
        node = self.root
        for ch in prefix:
            if ch not in node:
                return 0
            node = node[ch]
        return node["#"]

MapSumTemplate = MapSum
assert run_ops(MapSumTemplate, example_ops, example_args) == example_expected
rng = random.Random(20)
for _ in range(100):
    ops, args = gen_ops(rng, 50, "ab", 4, 1000)
    assert run_ops(MapSumTemplate, ops, args) == run_ops(Oracle, ["Oracle"] + ops[1:], args)
print("Template mục 20 OK")

Template mục 20 OK
